# Post-Hoc DyT in Pretrained Transformer Encoders

Companion notebook for the 6.s058 final paper. Repo: [zhizhen-kyle-luo/6s058-final-project](https://github.com/zhizhen-kyle-luo/6s058-final-project).

## 1. Install

In [ ]:
!pip install -q "transformers==4.46.3" "datasets>=3.2.0" "fsspec>=2025.3.0" "evaluate==0.4.3" "accelerate==1.1.1" "timm==1.0.11" 2>&1 | tail -n 3


In [ ]:
import torch, transformers, datasets, accelerate, timm
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("accelerate  ", accelerate.__version__)
print("timm        ", timm.__version__)
print("cuda        ", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


## 2. Repo + Drive

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/zhizhen-kyle-luo/6s058-final-project.git"
REPO_DIR = pathlib.Path("/content/6s058-final-project")

try:
    from google.colab import drive
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    FINAL_PAPER_DIR = REPO_DIR

    USE_DRIVE_FOR_OUTPUTS = True
    if USE_DRIVE_FOR_OUTPUTS:
        try:
            drive.mount("/content/drive", force_remount=False)
            DRIVE_OUTPUTS = pathlib.Path("/content/drive/MyDrive/6s058-final-project")
            DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
            RUNS_DIR    = DRIVE_OUTPUTS / "runs"
            FIGURES_DIR = DRIVE_OUTPUTS / "figures"
        except Exception as e:
            print(f"Drive mount failed ({e}); using ephemeral /content")
            RUNS_DIR    = FINAL_PAPER_DIR / "runs"
            FIGURES_DIR = FINAL_PAPER_DIR / "figures"
    else:
        RUNS_DIR    = FINAL_PAPER_DIR / "runs"
        FIGURES_DIR = FINAL_PAPER_DIR / "figures"
else:
    here = pathlib.Path.cwd()
    candidates = [here, *here.parents]
    FINAL_PAPER_DIR = next((p / "final_paper" for p in candidates if (p / "final_paper").exists()), None)
    if FINAL_PAPER_DIR is None:
        FINAL_PAPER_DIR = pathlib.Path("./final_paper")
        FINAL_PAPER_DIR.mkdir(parents=True, exist_ok=True)
    RUNS_DIR    = FINAL_PAPER_DIR / "runs"
    FIGURES_DIR = FINAL_PAPER_DIR / "figures"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("FINAL_PAPER_DIR =", FINAL_PAPER_DIR)
print("RUNS_DIR        =", RUNS_DIR)
print("FIGURES_DIR     =", FIGURES_DIR)
!nvidia-smi -L 2>/dev/null || echo "no nvidia-smi"


## 3. Config

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    task: str = "segmentation"
    variant: str = "ln"
    run_name: str = ""
    iters: int = 5000
    batch: int = 4
    crop: int = 512
    lr: float = 6e-5
    weight_decay: float = 0.01
    warmup_iters: int = 200
    eval_every: int = 500
    subset_train: int = 2000
    subset_val: int = 500
    seed: int = 42
    pretrained: str = "nvidia/mit-b0"
    num_classes: int = 150
    ignore_index: int = 255
    dyt_alpha_init: float = 0.5
    dyt_preserve_ln_affine: bool = False
    use_amp: bool = True

def cfg_classification(**overrides):
    base = dict(
        task="classification",
        iters=3000, batch=64, crop=224,
        lr=5e-5, weight_decay=0.01, warmup_iters=200, eval_every=500,
        pretrained="google/vit-base-patch16-224",
        num_classes=100, ignore_index=-100,
    )
    base.update(overrides)
    return Config(**base)

def cfg_segmentation(**overrides):
    return Config(**overrides)


## 4. DyT module + LayerNorm replacement

In [ ]:
import torch
import torch.nn as nn

class DyT(nn.Module):
    def __init__(self, normalized_shape, alpha_init: float = 0.5):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.alpha = nn.Parameter(torch.tensor(float(alpha_init)))
        self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        self.beta  = nn.Parameter(torch.zeros(self.normalized_shape))

    def forward(self, x):
        # NCHW first to disambiguate (B,C,H,C): treat 4D w/ x.shape[1]==C as NCHW
        if x.dim() == 4 and len(self.normalized_shape) == 1 and x.shape[1] == self.normalized_shape[0]:
            return self.gamma.view(1, -1, 1, 1) * torch.tanh(self.alpha * x) + self.beta.view(1, -1, 1, 1)
        if x.shape[-len(self.normalized_shape):] == self.normalized_shape:
            return self.gamma * torch.tanh(self.alpha * x) + self.beta
        raise RuntimeError(f"DyT shape mismatch: input {tuple(x.shape)} vs normalized_shape {self.normalized_shape}")


def _set_module(root, dotted, new):
    parts = dotted.split(".")
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def _auto_backbone_prefix(model):
    cls = model.__class__.__name__
    if "Segformer" in cls: return "segformer.encoder"
    if "ViT" in cls: return "vit.encoder"
    raise ValueError(f"unknown model class: {cls}; pass backbone_prefix explicitly")


def replace_ln_with_dyt(model, alpha_init=0.5, preserve_affine=False, backbone_prefix=None, verbose=True):
    if backbone_prefix is None:
        backbone_prefix = _auto_backbone_prefix(model)
    replaced = []
    for name, module in list(model.named_modules()):
        if not isinstance(module, nn.LayerNorm): continue
        if not name.startswith(backbone_prefix): continue
        params = list(module.parameters())
        device = params[0].device if params else torch.device("cpu")
        dtype = params[0].dtype if params else torch.float32
        new = DyT(module.normalized_shape, alpha_init=alpha_init).to(device=device, dtype=dtype)
        if preserve_affine:
            with torch.no_grad():
                new.gamma.copy_(module.weight)
                new.beta.copy_(module.bias)
        _set_module(model, name, new)
        replaced.append(name)
    if verbose:
        print(f"[replace_ln_with_dyt] preserve_affine={preserve_affine} replaced {len(replaced)} LayerNorms under {backbone_prefix}")
    return replaced


# inline check for the only new code path: preserve_affine actually copies weight/bias
def _check_affine_preserve():
    ln = nn.LayerNorm(32)
    with torch.no_grad():
        ln.weight.uniform_(0.5, 1.5); ln.bias.uniform_(-1.0, 1.0)
    container = nn.Sequential()
    container.add_module("vit", nn.Sequential())
    container.vit.add_module("encoder", nn.Sequential())
    container.vit.encoder.add_module("ln0", ln)
    rep = replace_ln_with_dyt(container, preserve_affine=True, backbone_prefix="vit.encoder", verbose=False)
    new = container.vit.encoder.ln0
    assert torch.allclose(new.gamma, ln.weight), "gamma must equal LN.weight"
    assert torch.allclose(new.beta, ln.bias), "beta must equal LN.bias"
    assert len(rep) == 1
_check_affine_preserve()


## 5. Saturation hooks + α logger

In [ ]:
from collections import defaultdict

class SaturationTracker:
    def __init__(self, model, threshold=0.95):
        self.threshold = threshold; self.values = defaultdict(list); self.handles = []
        for name, m in model.named_modules():
            if isinstance(m, DyT):
                self.handles.append(m.register_forward_hook(self._make_hook(name)))

    def _make_hook(self, name):
        def hook(module, inp, out):
            with torch.no_grad():
                pre = torch.tanh(module.alpha * inp[0])
                self.values[name].append((pre.abs() > self.threshold).float().mean().item())
        return hook

    def summary(self):
        return {k: float(sum(v) / max(len(v), 1)) for k, v in self.values.items()}

    def close(self):
        for h in self.handles: h.remove()


def collect_alphas(model):
    return {name: float(m.alpha.detach().cpu().item()) for name, m in model.named_modules() if isinstance(m, DyT)}


## 6. Data

In [ ]:
import random
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF

IM_MEAN = (0.485, 0.456, 0.406)
IM_STD  = (0.229, 0.224, 0.225)
# zhoubolei/scene_parse_150 still ships a loading script on main; use a Parquet duplicate.
SEGMENTATION_DATASET = "merve/scene_parse_150"
SEGMENTATION_COLUMNS = ("image", "annotation")


def _seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _check_segmentation_schema(ds):
    missing_splits = [split for split in ("train", "validation") if split not in ds]
    if missing_splits:
        raise KeyError(f"{SEGMENTATION_DATASET} is missing splits: {missing_splits}; found {list(ds.keys())}")
    for split in ("train", "validation"):
        missing_columns = [col for col in SEGMENTATION_COLUMNS if col not in ds[split].column_names]
        if missing_columns:
            raise KeyError(
                f"{SEGMENTATION_DATASET}/{split} is missing columns {missing_columns}; "
                f"found {ds[split].column_names}"
            )
    print(f"{SEGMENTATION_DATASET} schema OK", {split: ds[split].column_names for split in ("train", "validation")})


class ADE20KSubset(Dataset):
    def __init__(self, hf_split, indices, crop, train):
        self.ds = hf_split; self.indices = indices; self.crop = crop; self.train = train

    def __len__(self): return len(self.indices)

    def _remap(self, mask):
        # scene_parse_150 labels: 0=ignore, 1..150 -> 0..149
        m = mask.long()
        out = torch.full_like(m, 255)
        fg = m >= 1
        out[fg] = m[fg] - 1
        return out

    def __getitem__(self, i):
        ex = self.ds[int(self.indices[i])]
        img = ex["image"].convert("RGB"); msk = ex["annotation"]
        if self.train:
            scale = random.uniform(0.5, 2.0)
            new_size = (int(img.height * scale), int(img.width * scale))
            img = TF.resize(img, new_size, T.InterpolationMode.BILINEAR)
            msk = TF.resize(msk, new_size, T.InterpolationMode.NEAREST)
            pad_h = max(0, self.crop - img.height); pad_w = max(0, self.crop - img.width)
            if pad_h or pad_w:
                img = TF.pad(img, [0, 0, pad_w, pad_h], fill=0)
                msk = TF.pad(msk, [0, 0, pad_w, pad_h], fill=0)
            i0 = random.randint(0, img.height - self.crop)
            j0 = random.randint(0, img.width  - self.crop)
            img = TF.crop(img, i0, j0, self.crop, self.crop)
            msk = TF.crop(msk, i0, j0, self.crop, self.crop)
            if random.random() < 0.5:
                img = TF.hflip(img); msk = TF.hflip(msk)
            img = T.ColorJitter(0.4, 0.4, 0.4)(img)
        else:
            img = TF.resize(img, self.crop, T.InterpolationMode.BILINEAR)
            msk = TF.resize(msk, self.crop, T.InterpolationMode.NEAREST)
            img = TF.center_crop(img, self.crop)
            msk = TF.center_crop(msk, self.crop)
        img_t = TF.normalize(TF.to_tensor(img), IM_MEAN, IM_STD)
        msk_t = self._remap(torch.as_tensor(np.array(msk), dtype=torch.long))
        return img_t, msk_t


class CIFAR100Wrap(Dataset):
    def __init__(self, hf_split, crop, train):
        self.ds = hf_split; self.crop = crop; self.train = train

    def __len__(self): return len(self.ds)

    def __getitem__(self, i):
        ex = self.ds[int(i)]
        img = ex["img"].convert("RGB")
        label = int(ex["fine_label"])
        if self.train:
            img = TF.resize(img, self.crop, T.InterpolationMode.BILINEAR)
            img = T.RandomResizedCrop(self.crop, scale=(0.9, 1.0))(img)
            if random.random() < 0.5:
                img = TF.hflip(img)
        else:
            img = TF.resize(img, self.crop, T.InterpolationMode.BILINEAR)
            img = TF.center_crop(img, self.crop)
        img_t = TF.normalize(TF.to_tensor(img), IM_MEAN, IM_STD)
        return img_t, torch.tensor(label, dtype=torch.long)


def build_loaders(cfg):
    _seed_everything(cfg.seed)
    if cfg.task == "segmentation":
        ds = load_dataset(SEGMENTATION_DATASET)
        _check_segmentation_schema(ds)
        rng = np.random.RandomState(cfg.seed)
        train_idx = rng.choice(len(ds["train"]), size=cfg.subset_train, replace=False)
        val_idx   = rng.choice(len(ds["validation"]), size=cfg.subset_val, replace=False)
        train_ds = ADE20KSubset(ds["train"], train_idx, cfg.crop, train=True)
        val_ds   = ADE20KSubset(ds["validation"], val_idx, cfg.crop, train=False)
    elif cfg.task == "classification":
        ds = load_dataset("uoft-cs/cifar100")
        # CIFAR-100 has only train+test; we use HF "test" as validation
        train_ds = CIFAR100Wrap(ds["train"], cfg.crop, train=True)
        val_ds   = CIFAR100Wrap(ds["test"],  cfg.crop, train=False)
    else:
        raise ValueError(cfg.task)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch, shuffle=True,  num_workers=2, drop_last=True,  pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch, shuffle=False, num_workers=2, drop_last=False, pin_memory=True)
    return train_loader, val_loader


## 7. Build model

In [ ]:
from transformers import SegformerForSemanticSegmentation, ViTForImageClassification

def build_model(cfg):
    if cfg.task == "segmentation":
        model = SegformerForSemanticSegmentation.from_pretrained(
            cfg.pretrained, num_labels=cfg.num_classes, ignore_mismatched_sizes=True,
        )
    elif cfg.task == "classification":
        model = ViTForImageClassification.from_pretrained(
            cfg.pretrained, num_labels=cfg.num_classes, ignore_mismatched_sizes=True,
        )
    else:
        raise ValueError(cfg.task)
    if cfg.variant == "ln":
        return model, []
    if cfg.variant == "dyt_full":
        replaced = replace_ln_with_dyt(
            model,
            alpha_init=cfg.dyt_alpha_init,
            preserve_affine=cfg.dyt_preserve_ln_affine,
        )
        return model, replaced
    raise ValueError(cfg.variant)


## Smoke test

In [ ]:
def smoke_models():
    for cfg in [
        cfg_segmentation(task="segmentation", variant="dyt_full", run_name="smoke_seg",
                         subset_train=4, subset_val=4, batch=2, crop=128),
        cfg_classification(variant="dyt_full", run_name="smoke_vit", batch=2, iters=1),
    ]:
        loader, _ = build_loaders(cfg)
        xb, yb = next(iter(loader))
        model, replaced = build_model(cfg)
        assert len(replaced) > 0, f"{cfg.task}: replace_ln_with_dyt found 0 modules"
        out = model(pixel_values=xb).logits
        print(cfg.task, len(replaced), "replaced  out:", tuple(out.shape))
    print("smoke OK")

smoke_models()


## 8. Train loop

In [ ]:
import json, time, math


def _poly_lr(base, step, total, warmup, power=1.0):
    if step < warmup:
        return base * step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return base * (1 - p) ** power


def _cosine_lr(base, step, total, warmup):
    if step < warmup:
        return base * step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return base * 0.5 * (1.0 + math.cos(math.pi * p))


@torch.no_grad()
def run_eval(model, loader, cfg, device, use_amp):
    model.eval()
    if cfg.task == "segmentation":
        cm = torch.zeros(cfg.num_classes, cfg.num_classes, dtype=torch.long, device=device)
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with torch.cuda.amp.autocast(enabled=use_amp and device == "cuda"):
                out = model(pixel_values=xb).logits
                out = nn.functional.interpolate(out, size=yb.shape[-2:], mode="bilinear", align_corners=False)
            pred = out.argmax(1)
            valid = yb != cfg.ignore_index
            p = pred[valid]; t = yb[valid]
            idx = t * cfg.num_classes + p
            cm += torch.bincount(idx, minlength=cfg.num_classes * cfg.num_classes).reshape(cfg.num_classes, cfg.num_classes)
        cm = cm.cpu()
        inter = cm.diag().float()
        union = cm.sum(0).float() + cm.sum(1).float() - inter
        iou = torch.where(union > 0, inter / union, torch.full_like(inter, float("nan")))
        return {"primary": float(torch.nanmean(iou).item()), "metric_name": "miou"}
    else:
        correct = 0; total = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with torch.cuda.amp.autocast(enabled=use_amp and device == "cuda"):
                logits = model(pixel_values=xb).logits
            pred = logits.argmax(1)
            correct += (pred == yb).sum().item()
            total   += yb.numel()
        return {"primary": correct / max(1, total), "metric_name": "top1"}


def train_one(cfg, runs_dir):
    assert cfg.run_name, "Config.run_name is required"
    _seed_everything(cfg.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = cfg.use_amp and device == "cuda"

    run_dir = runs_dir / cfg.run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

    train_loader, val_loader = build_loaders(cfg)
    model, replaced = build_model(cfg)
    model = model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    lr_fn = _cosine_lr if cfg.task == "classification" else _poly_lr

    start_iter = 0
    best_primary = -1.0
    last_path = run_dir / "last.pt"
    best_path = run_dir / "best.pt"
    if last_path.exists():
        ck = torch.load(last_path, map_location=device)
        model.load_state_dict(ck["model"]); optim.load_state_dict(ck["optim"])
        if "scaler" in ck and use_amp and ck["scaler"] is not None:
            scaler.load_state_dict(ck["scaler"])
        start_iter = ck["iter"]; best_primary = ck.get("best_primary", -1.0)
        print(f"resumed iter={start_iter} best={best_primary:.4f}")

    log_path = run_dir / "log.jsonl"
    log_f = open(log_path, "a")
    train_iter = iter(train_loader)
    sat = SaturationTracker(model) if any(isinstance(m, DyT) for m in model.modules()) else None

    t0 = time.time()
    model.train()
    for step in range(start_iter, cfg.iters):
        try: xb, yb = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader); xb, yb = next(train_iter)
        xb, yb = xb.to(device), yb.to(device)
        lr = lr_fn(cfg.lr, step, cfg.iters, cfg.warmup_iters)
        for g in optim.param_groups: g["lr"] = lr
        optim.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            if cfg.task == "segmentation":
                out = model(pixel_values=xb).logits
                out = nn.functional.interpolate(out, size=yb.shape[-2:], mode="bilinear", align_corners=False)
                loss = nn.functional.cross_entropy(out, yb, ignore_index=cfg.ignore_index)
            else:
                out = model(pixel_values=xb).logits
                loss = nn.functional.cross_entropy(out, yb)
        scaler.scale(loss).backward()
        scaler.step(optim); scaler.update()

        if (step + 1) % 50 == 0:
            print(f"[{cfg.run_name}] step {step+1}/{cfg.iters}  loss={loss.item():.4f}  lr={lr:.2e}  elapsed={time.time()-t0:.0f}s")
            log_f.write(json.dumps({"kind": "train", "step": step+1, "loss": float(loss.item()), "lr": lr}) + "\n"); log_f.flush()

        if (step + 1) % cfg.eval_every == 0 or (step + 1) == cfg.iters:
            metrics = run_eval(model, val_loader, cfg, device, use_amp)
            sat_summary = sat.summary() if sat is not None else {}
            print(f"[{cfg.run_name}] eval @ {step+1}: {metrics['metric_name']}={metrics['primary']:.4f}")
            log_f.write(json.dumps({"kind": "eval", "step": step+1, **metrics, "saturation": sat_summary}) + "\n"); log_f.flush()
            if metrics["primary"] > best_primary:
                best_primary = metrics["primary"]
                torch.save({"model": model.state_dict(), "iter": step+1, "primary": best_primary}, best_path)
            torch.save({
                "model": model.state_dict(), "optim": optim.state_dict(),
                "scaler": scaler.state_dict() if use_amp else None,
                "iter": step+1, "best_primary": best_primary,
            }, last_path)
            model.train()

    log_f.close()
    if sat is not None: sat.close()

    summary = {
        "run_name": cfg.run_name,
        "task": cfg.task,
        "variant": cfg.variant,
        "data_source": SEGMENTATION_DATASET if cfg.task == "segmentation" else "uoft-cs/cifar100",
        "iters": cfg.iters,
        "best_primary": best_primary,
        "metric_name": "miou" if cfg.task == "segmentation" else "top1",
        "num_replaced": len(replaced),
        "replaced_modules": replaced,
        "final_alphas": collect_alphas(model),
        "final_saturation": sat.summary() if sat is not None else {},
        "dyt_alpha_init": cfg.dyt_alpha_init,
        "dyt_preserve_ln_affine": cfg.dyt_preserve_ln_affine,
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    print("DONE", cfg.run_name, "best=", best_primary)
    return summary


## 9. Runs

Before comparing segmentation variants, clear any older segmentation checkpoints trained from the original scripted dataset loader. The three segmentation cells below should all run against the same `merve/scene_parse_150` Parquet ordering.

In [ ]:
import shutil

SEGMENTATION_RUNS = ("seg_ln", "seg_dyt_full_fresh", "seg_dyt_full_affine")

def clear_segmentation_runs(runs_dir=RUNS_DIR):
    for name in SEGMENTATION_RUNS:
        path = runs_dir / name
        if path.exists():
            shutil.rmtree(path)
            print("removed", path)
        else:
            print("missing", path)

# Run once before the segmentation trio if Drive contains old scripted-dataset checkpoints.
# clear_segmentation_runs()


### `seg_ln` — control (existing)

In [ ]:
train_one(cfg_segmentation(task="segmentation", variant="ln", run_name="seg_ln"), RUNS_DIR)


### `seg_dyt_full_fresh` — paper-default in post-hoc (existing)

In [ ]:
train_one(cfg_segmentation(task="segmentation", variant="dyt_full", run_name="seg_dyt_full_fresh"), RUNS_DIR)


### `seg_dyt_full_affine` — proposed init

In [ ]:
train_one(cfg_segmentation(task="segmentation", variant="dyt_full", run_name="seg_dyt_full_affine", dyt_preserve_ln_affine=True), RUNS_DIR)


### `vit_ln` — classification control (soft sanity: top-1 ≥ 60%)

In [ ]:
train_one(cfg_classification(variant="ln", run_name="vit_ln"), RUNS_DIR)


### `vit_dyt_full_fresh` — paper-default in classification post-hoc

In [ ]:
train_one(cfg_classification(variant="dyt_full", run_name="vit_dyt_full_fresh"), RUNS_DIR)


### `vit_dyt_full_affine` — proposed init in classification

In [ ]:
train_one(cfg_classification(variant="dyt_full", run_name="vit_dyt_full_affine", dyt_preserve_ln_affine=True), RUNS_DIR)


## 10. Diagnostics

In [ ]:
import matplotlib.pyplot as plt

def _read_jsonl(p):
    return [json.loads(l) for l in open(p)]


def _runs_for_task(task):
    return [d.name for d in RUNS_DIR.iterdir() if d.is_dir() and (d / "summary.json").exists()
            and json.loads((d / "summary.json").read_text()).get("task") == task]


def plot_curves(task, out_path):
    fig_l, ax_l = plt.subplots(figsize=(5, 3))
    fig_m, ax_m = plt.subplots(figsize=(5, 3))
    metric_name = "miou" if task == "segmentation" else "top1"
    for run in _runs_for_task(task):
        log = _read_jsonl(RUNS_DIR / run / "log.jsonl")
        ts = [r for r in log if r["kind"] == "train"]
        es = [r for r in log if r["kind"] == "eval"]
        ax_l.plot([r["step"] for r in ts], [r["loss"] for r in ts], label=run)
        ax_m.plot([r["step"] for r in es], [r["primary"] for r in es], marker="o", label=run)
    for ax, ttl, ylab in [(ax_l, f"{task} training loss", "loss"), (ax_m, f"{task} validation {metric_name}", metric_name)]:
        ax.set_xlabel("iter"); ax.set_ylabel(ylab); ax.set_title(ttl); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig_l.tight_layout(); fig_m.tight_layout()
    fig_l.savefig(out_path / f"{task}_loss.pdf")
    fig_m.savefig(out_path / f"{task}_{metric_name}.pdf")


def plot_alpha_drift(task, out_path):
    fig, ax = plt.subplots(figsize=(8, 3))
    for run in _runs_for_task(task):
        s = json.loads((RUNS_DIR / run / "summary.json").read_text())
        alphas = s.get("final_alphas", {})
        if not alphas: continue
        names = list(alphas.keys())
        vals = [alphas[n] for n in names]
        ax.plot(range(len(vals)), vals, marker=".", label=run)
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.7, label="α₀=0.5")
    ax.set_xlabel("DyT layer index"); ax.set_ylabel("trained α")
    ax.set_title(f"{task}: trained α per layer"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(out_path / f"{task}_alpha.pdf")


def plot_saturation(task, out_path):
    fig, ax = plt.subplots(figsize=(8, 3))
    for run in _runs_for_task(task):
        s = json.loads((RUNS_DIR / run / "summary.json").read_text())
        sat = s.get("final_saturation", {})
        if not sat: continue
        names = list(sat.keys())
        vals = [sat[n] for n in names]
        ax.plot(range(len(vals)), vals, marker=".", label=run)
    ax.set_xlabel("DyT layer index"); ax.set_ylabel("frac |tanh(αx)| > 0.95")
    ax.set_title(f"{task}: saturation per layer"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(out_path / f"{task}_saturation.pdf")


for task in ["segmentation", "classification"]:
    plot_curves(task, FIGURES_DIR)
    plot_alpha_drift(task, FIGURES_DIR)
    plot_saturation(task, FIGURES_DIR)


## 11. Summary table

In [ ]:
rows = []
for d in sorted(RUNS_DIR.iterdir()):
    p = d / "summary.json"
    if p.exists():
        s = json.loads(p.read_text())
        rows.append({
            "run": s.get("run_name", d.name),
            "task": s.get("task"),
            "variant": s.get("variant"),
            "preserve_affine": s.get("dyt_preserve_ln_affine"),
            "metric": s.get("metric_name"),
            "best": round(s.get("best_primary", -1), 4),
            "n_replaced": s.get("num_replaced"),
        })
print(json.dumps(rows, indent=2))
(FIGURES_DIR / "table_main.json").write_text(json.dumps(rows, indent=2))
